In [1]:
# ==============================================================================
# EJUST Egyptian Arabic ASR — Q1 Missing Experiments & Fast Dual-T4 Runner
# - Skips already-completed conmamba_small (seed 7775: 13.85 / 14.65 WER)
# - Runs missing baselines (conformer_small, conmambamamba_small) + multi-seed
# - Evaluates 3 long-form decoding strategies (fixed, chunked, overlap_energy)
# - Optional fast Whisper baseline on test / test_long
# - Prints full Q1 tables, RTF, Multi-Seed Mean ± Std, and LaTeX (no figures)
# ==============================================================================

import os, sys, re, glob, json, time, shutil, platform, subprocess
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

# ==============================================================================
# 0. EXPERIMENT CONFIGURATION (EDIT ONLY THIS BLOCK)
# ==============================================================================
EXPERIMENTS = [
    # (model_name, seed)
    ("conformer_small",     7775),
    ("conmambamamba_small", 7775),
    # Uncomment for full 3-seed Q1 statistical validation (runs 2 at a time on T4x2):
    # ("conformer_small",     42),
    # ("conmambamamba_small", 42),
    # ("conmamba_small",      42),
    # ("conformer_small",     1234),
    # ("conmambamamba_small", 1234),
    # ("conmamba_small",      1234),
]

EPOCHS = 110                    # Zevallos et al. (2025)
VALID_SEARCH_INTERVAL = 55      # Fast: beam-search valid only at ep 55 & 110
TIME_LIMIT_H = 11.2             # Safe stop before Kaggle's 12h hard limit
RUN_WHISPER_BASELINE = True     # Fast (~4 min) inference-only Whisper baseline
WHISPER_MODEL = "openai/whisper-small"  # Or "openai/whisper-large-v3"
LONG_STRATEGIES = ["fixed", "chunked", "overlap_energy"]  # Q1 Option 2 ablation
PREV_CONMAMBA_RESULT = {        # Already-completed run (included in final Q1 table)
    "model": "conmamba_small", "seed": 7775, "params_m": 12.1,
    "test_wer": 13.85, "test_cer": 4.44, "test_long_wer": 14.65, "test_long_cer": 4.57,
    "test_ctc_wer": 18.83, "test_long_ctc_wer": 72.94, "rtf_test": None, "rtf_long": None,
}
RESUME_INPUT_DIR = None         # Optional: "/kaggle/input/<prev-run-dataset>"

WORK = "/kaggle/working"
REPO = f"{WORK}/Mamba-ASR"
RESULTS_ROOT = f"{WORK}/q1_results"
os.makedirs(RESULTS_ROOT, exist_ok=True)

# ==============================================================================
# 1. ENVIRONMENT & DEPENDENCY SETUP
# ==============================================================================
import torch
NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS >= 1, "No GPU detected! Enable Accelerator -> GPU T4 x2."
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | GPUs: {NUM_GPUS}x {torch.cuda.get_device_name(0)}")

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

pip_install(["speechbrain==1.0.3", "sentencepiece", "einops", "soundfile", "packaging", "ninja", "jiwer", "scipy"])
pip_install(["causal-conv1d", "mamba-ssm", "--no-build-isolation"])
if RUN_WHISPER_BASELINE:
    pip_install(["transformers", "accelerate"])

if not os.path.isdir(REPO):
    subprocess.check_call(["git", "clone", "-q", "https://github.com/xi-j/Mamba-ASR.git", REPO])

# ==============================================================================
# 2. WRITE PATCHED ejust_tools.py (FIXED YAML + SPEED + Q1 OVERRIDES)
# ==============================================================================
ejust_tools_code = r'''
from pathlib import Path
import re

SHIM = """# --- compatibility shim (EJUST) ---
try:
    from causal_conv1d.cpp_functions import (
        causal_conv1d_fwd_function as _cc_new_fwd,
        causal_conv1d_bwd_function as _cc_new_bwd,
    )
    def _cc_fwd(x, weight, bias, seq_idx, silu):
        return _cc_new_fwd(x, weight, bias, seq_idx, None, None, silu)
    def _cc_bwd(x, weight, bias, dout, seq_idx, dx, silu):
        dx, dweight, dbias, _ = _cc_new_bwd(x, weight, bias, dout, seq_idx, None, None, dx, False, silu)
        return dx, dweight, dbias
except ImportError:
    import causal_conv1d_cuda
    def _cc_fwd(x, weight, bias, seq_idx, silu):
        try:
            return causal_conv1d_cuda.causal_conv1d_fwd(x, weight, bias, seq_idx, None, None, silu)
        except TypeError:
            return causal_conv1d_cuda.causal_conv1d_fwd(x, weight, bias, seq_idx, silu)
    def _cc_bwd(x, weight, bias, dout, seq_idx, dx, silu):
        try:
            r = causal_conv1d_cuda.causal_conv1d_bwd(x, weight, bias, dout, seq_idx, None, None, dx, False, silu)
            return r[0], r[1], r[2]
        except TypeError:
            return causal_conv1d_cuda.causal_conv1d_bwd(x, weight, bias, dout, seq_idx, dx, silu)
"""

LN_OLD = "    from mamba_ssm.ops.triton.layernorm import RMSNorm, layer_norm_fn, rms_norm_fn\n"
LN_NEW = (
    "    try:\n"
    "        from mamba_ssm.ops.triton.layernorm import RMSNorm, layer_norm_fn, rms_norm_fn\n"
    "    except ImportError:\n"
    "        from mamba_ssm.ops.triton.layer_norm import RMSNorm, layer_norm_fn, rms_norm_fn\n"
)

def patch_repo(repo):
    repo = Path(repo)
    p = repo / "modules/mamba/selective_scan_interface.py"
    s = p.read_text()
    if "_cc_fwd" not in s:
        s = re.sub(r"causal_conv1d_cuda\.causal_conv1d_fwd\((x, conv1d_weight)", r"_cc_fwd(\1", s)
        s = s.replace("causal_conv1d_cuda.causal_conv1d_bwd(\n", "_cc_bwd(\n")
        s = s.replace("import causal_conv1d_cuda\n", SHIM, 1)
        p.write_text(s)
    for f in ["modules/mamba/bimamba.py", "modules/mamba/mamba_blocks.py"]:
        p = repo / f
        s = p.read_text()
        if "except ImportError:" not in s and LN_OLD in s:
            p.write_text(s.replace(LN_OLD, LN_NEW, 1))
    p_tasr = repo / "modules/TransformerASR.py"
    if p_tasr.exists():
        s_tasr = p_tasr.read_text()
        old_tgt = "if self.num_decoder_layers > 0:\n            tgt = self.custom_tgt_module(tgt)"
        new_tgt = "if self.num_decoder_layers > 0 and tgt is not None:\n            tgt = self.custom_tgt_module(tgt)"
        if old_tgt in s_tasr:
            p_tasr.write_text(s_tasr.replace(old_tgt, new_tgt, 1))
    return True

def make_hparams(template, out_path, *, data_folder, csv_dir, bpe_model, output_folder,
                 epochs, warmup, valid_search_interval, batch_size=32, lr=0.0008,
                 num_workers=4, grad_accumulation_factor=1, long_strategies="overlap_energy",
                 skip_train=False):
    s = Path(template).read_text()
    rep = [
        (r"^data_folder: !PLACEHOLDER.*$",
         f"data_folder: {data_folder}\nbpe_model: {bpe_model}\nlong_strategies: {long_strategies}"),
        (r"^skip_train: .*$", f"skip_train: {skip_train}"),
        (r"^output_folder: .*$", f"output_folder: {output_folder}"),
        (r"^train_csv: .*$", f"train_csv: {csv_dir}/train.csv"),
        (r"^valid_csv: .*$", f"valid_csv: {csv_dir}/dev.csv"),
        (r"^test_csv:\n(    - .*\n)+",
         f"test_csv:\n    - {csv_dir}/test.csv\n    - {csv_dir}/test_long.csv\n"),
        (r"^number_of_epochs: .*$", f"number_of_epochs: {epochs}"),
        (r"^grad_accumulation_factor: .*$", f"grad_accumulation_factor: {grad_accumulation_factor}"),
        (r"^num_workers: .*$", f"num_workers: {num_workers}"),
        (r"^precision: .*$", "precision: fp16"),
        (r"^lr_adam: .*$", f"lr_adam: {lr}"),
        (r"^batch_size: .*$", f"batch_size: {batch_size}"),
        (r"^dynamic_batching: .*$", "dynamic_batching: False"),
        (r"^valid_dataloader_opts:\n    batch_size: 1$", "valid_dataloader_opts:\n    batch_size: 32"),
        (r"^test_dataloader_opts:\n    batch_size: 1$", "test_dataloader_opts:\n    batch_size: 16"),
        (r"^output_neurons: .*$", "output_neurons: 500"),
        (r"^valid_search_interval: .*$", f"valid_search_interval: {valid_search_interval}"),
        (r"^test_beam_size: .*$", "test_beam_size: 10"),
        (r"^    n_warmup_steps: .*$", f"    n_warmup_steps: {warmup}"),
    ]
    for pat, new in rep:
        s, n = re.subn(pat, new, s, count=1, flags=re.M)
        assert n == 1, f"{Path(template).name}: pattern not found: {pat}"
    Path(out_path).write_text(s)
    return out_path
'''
Path(f"{WORK}/ejust_tools.py").write_text(ejust_tools_code)
sys.path.insert(0, WORK); sys.path.insert(0, REPO)
import ejust_tools
ejust_tools.patch_repo(REPO)
print("Patched Mamba-ASR successfully.")

# ==============================================================================
# 3. WRITE Q1-GRADE train_egy_q1.py (BATCHED CHUNKING + OVERLAP MERGE + RTF)
# ==============================================================================
train_script = r'''#!/usr/bin/env python3
import os, sys, json, time, logging
from pathlib import Path
import torch
import soundfile as sf
import torchaudio

if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]

torch.backends.cudnn.benchmark = True
import speechbrain as sb
from hyperpyyaml import load_hyperpyyaml
from speechbrain.utils.distributed import if_main_process

logger = logging.getLogger(__name__)

# --- Long-Form Segmentation Strategies (Q1 Option 2) ---
def segment_fixed(wav, sr=16000, max_dur=4.0):
    max_s = int(max_dur * sr)
    return [wav[i:i + max_s] for i in range(0, wav.size(-1), max_s)]

def segment_energy(wav, sr=16000, max_dur=4.0, min_dur=1.5, overlap_dur=0.0, hop_size=160, frame_size=400):
    total = wav.size(-1)
    max_s, min_s, ov_s = int(max_dur * sr), int(min_dur * sr), int(overlap_dur * sr)
    if total <= max_s or total < frame_size:
        return [wav]
    unfolded = wav.unfold(-1, frame_size, hop_size)
    energy = torch.sqrt(torch.mean(unfolded ** 2, dim=-1) + 1e-12)
    chunks, start = [], 0
    while start < total:
        if total - start <= max_s:
            chunks.append(wav[start:])
            break
        f_start = (start + min_s) // hop_size
        f_end = min((start + max_s) // hop_size, energy.size(-1))
        if f_end <= f_start:
            split = min(start + max_s, total)
        else:
            split = (f_start + torch.argmin(energy[f_start:f_end]).item()) * hop_size
        if split <= start:
            split = min(start + max_s, total)
        chunks.append(wav[start:split])
        next_start = max(split - ov_s, start + min_s // 2)
        start = next_start if ov_s > 0 else split
    return chunks

def merge_overlapping_texts(texts, max_ov_words=5):
    """Deduplicate repeated boundary words across overlapping chunks."""
    if not texts:
        return ""
    merged = texts[0].split()
    for txt in texts[1:]:
        words = txt.split()
        if not words:
            continue
        best_k = 0
        limit = min(max_ov_words, len(merged), len(words))
        for k in range(limit, 0, -1):
            if merged[-k:] == words[:k]:
                best_k = k
                break
        merged.extend(words[best_k:])
    return " ".join(merged)

class ASR(sb.core.Brain):
    def _decode_chunks_batched(self, chunks, current_epoch, searcher, batch_size=16):
        """Fast batched chunk decoding on GPU (4-6x faster than sequential loop)."""
        valid_chunks = [c for c in chunks if c.size(0) >= 400]
        if not valid_chunks:
            return []
        texts = []
        for i in range(0, len(valid_chunks), batch_size):
            sub = valid_chunks[i:i + batch_size]
            max_l = max(c.size(0) for c in sub)
            padded = torch.zeros(len(sub), max_l, device=self.device)
            lens = torch.zeros(len(sub), device=self.device)
            for j, c in enumerate(sub):
                padded[j, :c.size(0)] = c
                lens[j] = c.size(0) / max_l
            feats = self.hparams.compute_features(padded)
            feats = self.modules.normalize(feats, lens, epoch=current_epoch)
            src = self.modules.CNN(feats)
            if hasattr(self.modules.Transformer, "encode"):
                enc = self.modules.Transformer.encode(src, wav_len=lens)
            else:
                enc, _ = self.modules.Transformer(src, None, lens)
            hyps, _, _, _ = searcher(enc.detach(), lens)
            for h in hyps:
                t = self.tokenizer.decode_ids(h).strip()
                if t:
                    texts.append(t)
        return texts

    def decode_long_utterance(self, wav, current_epoch, searcher, strategy):
        sr = self.hparams.sample_rate
        if strategy == "fixed":
            chunks = segment_fixed(wav, sr=sr, max_dur=4.0)
            texts = self._decode_chunks_batched(chunks, current_epoch, searcher)
            full_text = " ".join(texts).strip()
        elif strategy == "overlap_energy":
            chunks = segment_energy(wav, sr=sr, max_dur=4.0, min_dur=1.5, overlap_dur=0.5)
            texts = self._decode_chunks_batched(chunks, current_epoch, searcher)
            full_text = merge_overlapping_texts(texts)
        else:
            chunks = segment_energy(wav, sr=sr, max_dur=4.0, min_dur=1.5, overlap_dur=0.0)
            texts = self._decode_chunks_batched(chunks, current_epoch, searcher)
            full_text = " ".join(texts).strip()
        return self.tokenizer.encode_as_ids(full_text)

    def compute_forward(self, batch, stage):
        batch = batch.to(self.device)
        wavs, wav_lens = batch.sig
        tokens_bos, _ = batch.tokens_bos
        if stage == sb.Stage.TEST:
            self._test_audio_sec += sum(
                wav_lens[k].item() * wavs.size(1) / self.hparams.sample_rate
                for k in range(wavs.size(0))
            )

        feats = self.hparams.compute_features(wavs)
        current_epoch = self.hparams.epoch_counter.current
        feats = self.modules.normalize(feats, wav_lens, epoch=current_epoch)
        if stage == sb.Stage.TRAIN and hasattr(self.hparams, "fea_augment"):
            feats, fea_lens = self.hparams.fea_augment(feats, wav_lens)
            tokens_bos = self.hparams.fea_augment.replicate_labels(tokens_bos)
        src = self.modules.CNN(feats)
        enc_out, pred = self.modules.Transformer(src, tokens_bos, wav_lens, pad_idx=self.hparams.pad_index)
        p_ctc = self.hparams.log_softmax(self.modules.ctc_lin(enc_out))
        p_seq = self.hparams.log_softmax(self.modules.seq_lin(pred))

        hyps = None
        is_valid_search = stage == sb.Stage.VALID and (
            current_epoch % self.hparams.valid_search_interval == 0
            or current_epoch == self.hparams.number_of_epochs
        )
        if is_valid_search:
            hyps, _, _, _ = self.hparams.valid_search(enc_out.detach(), wav_lens)
        elif stage == sb.Stage.TEST:
            searcher = self.hparams.test_search
            strategy = getattr(self.hparams, "active_long_strategy", "chunked")
            sr = self.hparams.sample_rate
            max_dur = max(wav_lens[k].item() * wavs.size(1) / sr for k in range(wavs.size(0)))
            if max_dur > 4.5 and strategy in ("fixed", "chunked", "overlap_energy"):
                hyps = []
                for b in range(wavs.size(0)):
                    b_samp = int(round(wav_lens[b].item() * wavs.size(1)))
                    b_wav = wavs[b, :b_samp]
                    hyps.append(self.decode_long_utterance(b_wav, current_epoch, searcher, strategy))
            else:
                hyps, _, _, _ = searcher(enc_out.detach(), wav_lens)
        return p_ctc, p_seq, wav_lens, hyps

    def compute_objectives(self, predictions, batch, stage):
        p_ctc, p_seq, wav_lens, hyps = predictions
        ids = batch.id
        tokens_eos, tokens_eos_lens = batch.tokens_eos
        tokens, tokens_lens = batch.tokens
        if stage == sb.Stage.TRAIN and hasattr(self.hparams, "fea_augment"):
            rep = self.hparams.fea_augment.replicate_labels
            tokens, tokens_lens = rep(tokens), rep(tokens_lens)
            tokens_eos, tokens_eos_lens = rep(tokens_eos), rep(tokens_eos_lens)
        loss_seq = self.hparams.seq_cost(p_seq, tokens_eos, length=tokens_eos_lens).sum()
        loss_ctc = self.hparams.ctc_cost(p_ctc, tokens, wav_lens, tokens_lens).sum()
        loss = self.hparams.ctc_weight * loss_ctc + (1 - self.hparams.ctc_weight) * loss_seq
        if stage != sb.Stage.TRAIN:
            current_epoch = self.hparams.epoch_counter.current
            do_wer = (
                stage == sb.Stage.TEST
                or current_epoch % self.hparams.valid_search_interval == 0
                or current_epoch == self.hparams.number_of_epochs
            )
            if do_wer and hyps is not None:
                predicted_words = [self.tokenizer.decode_ids(h).split(" ") for h in hyps]
                target_words = [w.split(" ") for w in batch.wrd]
                self.wer_metric.append(ids, predicted_words, target_words)
                self.cer_metric.append(ids, predicted_words, target_words)
                if stage == sb.Stage.TEST and hasattr(self, "ctc_wer_metric"):
                    ctc_words = []
                    argmax_toks = torch.argmax(p_ctc, dim=-1)
                    for b in range(p_ctc.size(0)):
                        v_len = int(round(wav_lens[b].item() * p_ctc.size(1)))
                        seq = argmax_toks[b, :v_len].tolist()
                        collapsed = [k for j, k in enumerate(seq) if j == 0 or k != seq[j - 1]]
                        final_toks = [k for k in collapsed if k != self.hparams.blank_index]
                        c_text = self.tokenizer.decode_ids(final_toks).strip()
                        ctc_words.append(c_text.split(" ") if c_text else [])
                    self.ctc_wer_metric.append(ids, ctc_words, target_words)
                    self.ctc_cer_metric.append(ids, ctc_words, target_words)
            self.acc_metric.append(p_seq, tokens_eos, tokens_eos_lens)
        return loss

    def on_evaluate_start(self, max_key=None, min_key=None):
        super().on_evaluate_start()
        ckpts = self.checkpointer.find_checkpoints(max_key=max_key, min_key=min_key)
        ckpt = sb.utils.checkpoints.average_checkpoints(ckpts, recoverable_name="model")
        self.hparams.model.load_state_dict(ckpt, strict=True)
        self.hparams.model.eval()

    def on_stage_start(self, stage, epoch):
        self._t0 = time.time()
        self._test_audio_sec = 0.0
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        if stage != sb.Stage.TRAIN:
            self.acc_metric = self.hparams.acc_computer()
            self.wer_metric = self.hparams.error_rate_computer()
            self.cer_metric = self.hparams.error_rate_computer(split_tokens=True)
            if stage == sb.Stage.TEST:
                self.ctc_wer_metric = self.hparams.error_rate_computer()
                self.ctc_cer_metric = self.hparams.error_rate_computer(split_tokens=True)

    def _log_efficiency(self, stage, epoch):
        if not if_main_process():
            return
        elapsed = round(time.time() - self._t0, 2)
        rtf = round(elapsed / self._test_audio_sec, 4) if (stage == sb.Stage.TEST and self._test_audio_sec > 0) else None
        rec = {
            "stage": str(stage).split(".")[-1],
            "epoch": epoch,
            "seconds": elapsed,
            "audio_sec": round(self._test_audio_sec, 1) if stage == sb.Stage.TEST else None,
            "rtf": rtf,
            "peak_gpu_mb": round(torch.cuda.max_memory_allocated() / 2**20) if torch.cuda.is_available() else None,
            "test_name": getattr(self.hparams, "test_name", None) if stage == sb.Stage.TEST else None,
        }
        with open(os.path.join(self.hparams.output_folder, "efficiency.jsonl"), "a") as f:
            f.write(json.dumps(rec) + "\n")

    def on_stage_end(self, stage, stage_loss, epoch):
        self._log_efficiency(stage, epoch)
        stage_stats = {"loss": stage_loss}
        if stage == sb.Stage.TRAIN:
            self.train_stats = stage_stats
            return
        stage_stats["ACC"] = self.acc_metric.summarize()
        current_epoch = self.hparams.epoch_counter.current
        do_wer = (
            stage == sb.Stage.TEST
            or current_epoch % self.hparams.valid_search_interval == 0
            or current_epoch == self.hparams.number_of_epochs
        )
        if do_wer:
            stage_stats["WER"] = self.wer_metric.summarize("error_rate")
            stage_stats["CER"] = self.cer_metric.summarize("error_rate")
            if stage == sb.Stage.TEST and hasattr(self, "ctc_wer_metric"):
                stage_stats["CTC_WER"] = self.ctc_wer_metric.summarize("error_rate")
                stage_stats["CTC_CER"] = self.ctc_cer_metric.summarize("error_rate")
        if stage == sb.Stage.VALID:
            lr = self.hparams.noam_annealing.current_lr
            self.hparams.train_logger.log_stats(
                stats_meta={"epoch": epoch, "lr": lr, "steps": self.optimizer_step},
                train_stats=self.train_stats, valid_stats=stage_stats,
            )
            self.checkpointer.save_and_keep_only(
                meta={"ACC": stage_stats["ACC"], "epoch": epoch},
                max_keys=["ACC"], num_to_keep=self.hparams.avg_checkpoints,
            )
        elif stage == sb.Stage.TEST:
            self.hparams.train_logger.log_stats(
                stats_meta={"Epoch": self.hparams.epoch_counter.current, "test": self.hparams.test_name},
                test_stats=stage_stats,
            )
            if if_main_process():
                with open(self.hparams.test_wer_file, "w", encoding="utf-8") as w:
                    self.wer_metric.write_stats(w)
                with open(self.hparams.test_wer_file.replace("wer_", "cer_"), "w", encoding="utf-8") as w:
                    self.cer_metric.write_stats(w)
                with open(self.hparams.test_wer_file.replace("wer_", "wer_ctc_"), "w", encoding="utf-8") as w:
                    self.ctc_wer_metric.write_stats(w)
                with open(self.hparams.test_wer_file.replace("wer_", "cer_ctc_"), "w", encoding="utf-8") as w:
                    self.ctc_cer_metric.write_stats(w)

    def on_fit_batch_end(self, batch, outputs, loss, should_step):
        if should_step:
            self.hparams.noam_annealing(self.optimizer)

def dataio_prepare(hparams, tokenizer):
    data_folder = hparams["data_folder"]
    load = lambda p: sb.dataio.dataset.DynamicItemDataset.from_csv(csv_path=p, replacements={"data_root": data_folder})
    train_data = load(hparams["train_csv"])
    valid_data = load(hparams["valid_csv"]).filtered_sorted(sort_key="duration")
    test_datasets = {Path(p).stem: load(p).filtered_sorted(sort_key="duration") for p in hparams["test_csv"]}
    datasets = [train_data, valid_data] + list(test_datasets.values())
    valtest = [valid_data] + list(test_datasets.values())

    def read(wav):
        x, sr = sf.read(wav, dtype="float32", always_2d=False)
        return torch.from_numpy(x)

    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def audio_pipeline(wav):
        return read(wav)
    sb.dataio.dataset.add_dynamic_item(valtest, audio_pipeline)

    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def audio_pipeline_train(wav):
        sig = read(wav)
        if "speed_perturb" in hparams:
            sig = hparams["speed_perturb"](sig.unsqueeze(0)).squeeze(0)
        return sig
    sb.dataio.dataset.add_dynamic_item([train_data], audio_pipeline_train)

    @sb.utils.data_pipeline.takes("wrd")
    @sb.utils.data_pipeline.provides("wrd", "tokens_list", "tokens_bos", "tokens_eos", "tokens")
    def text_pipeline(wrd):
        yield wrd
        tokens_list = tokenizer.encode_as_ids(wrd)
        yield tokens_list
        yield torch.LongTensor([hparams["bos_index"]] + tokens_list)
        yield torch.LongTensor(tokens_list + [hparams["eos_index"]])
        yield torch.LongTensor(tokens_list)
    sb.dataio.dataset.add_dynamic_item(datasets, text_pipeline)
    sb.dataio.dataset.set_output_keys(datasets, ["id", "sig", "wrd", "tokens_bos", "tokens_eos", "tokens"])
    return train_data, valid_data, test_datasets

if __name__ == "__main__":
    hparams_file, run_opts, overrides = sb.parse_arguments(sys.argv[1:])
    with open(hparams_file, encoding="utf-8") as fin:
        hparams = load_hyperpyyaml(fin, overrides)
    sb.utils.distributed.ddp_init_group(run_opts)
    sb.create_experiment_directory(
        experiment_directory=hparams["output_folder"],
        hyperparams_to_save=hparams_file, overrides=overrides,
    )
    tokenizer = hparams["tokenizer"]
    tokenizer.load(hparams["bpe_model"])
    train_data, valid_data, test_datasets = dataio_prepare(hparams, tokenizer)
    hparams["test_search"] = hparams["valid_search"]
    hparams["test_search"].beam_size = hparams["test_beam_size"]

    asr_brain = ASR(
        modules=hparams["modules"], opt_class=hparams["Adam"],
        hparams=hparams, run_opts=run_opts, checkpointer=hparams["checkpointer"],
    )
    asr_brain.tokenizer = tokenizer

    n_params = sum(p.numel() for p in asr_brain.hparams.model.parameters())
    Path(hparams["output_folder"], "params.json").write_text(json.dumps({"params": n_params}))

    if not hparams["skip_train"]:
        asr_brain.fit(
            asr_brain.hparams.epoch_counter, train_data, valid_data,
            train_loader_kwargs=hparams["train_dataloader_opts"],
            valid_loader_kwargs=hparams["valid_dataloader_opts"],
        )

    os.makedirs(hparams["output_wer_folder"], exist_ok=True)
    strategies = [s.strip() for s in str(hparams.get("long_strategies", "overlap_energy")).split(",")]

    for name, data in test_datasets.items():
        if "long" in name:
            for strat in strategies:
                asr_brain.hparams.active_long_strategy = strat
                asr_brain.hparams.test_name = f"{name}_{strat}"
                asr_brain.hparams.test_wer_file = os.path.join(hparams["output_wer_folder"], f"wer_{name}_{strat}.txt")
                test_opts = dict(hparams["test_dataloader_opts"])
                test_opts["batch_size"] = 1
                asr_brain.evaluate(data, max_key="ACC", test_loader_kwargs=test_opts)
        else:
            asr_brain.hparams.active_long_strategy = "standard"
            asr_brain.hparams.test_name = name
            asr_brain.hparams.test_wer_file = os.path.join(hparams["output_wer_folder"], f"wer_{name}.txt")
            asr_brain.evaluate(data, max_key="ACC", test_loader_kwargs=dict(hparams["test_dataloader_opts"]))
'''
Path(f"{REPO}/train_egy_q1.py").write_text(train_script)

# ==============================================================================
# 4. LOCATE DATASET (ejust-prepared-v1)
# ==============================================================================
import pandas as pd
bpe_candidates = sorted(glob.glob("/kaggle/input/**/tokenizer/bpe500.model", recursive=True))
assert bpe_candidates, "ejust-prepared-v1 not found! Add Input -> Datasets -> ejust-prepared-v1."
BPE = bpe_candidates[0]
DATA = os.path.dirname(os.path.dirname(BPE))
print(f"Dataset root: {DATA}")

# ==============================================================================
# 5. PARALLEL DUAL-GPU TRAINING RUNNER (GPU 0 & GPU 1 SIMULTANEOUSLY)
# ==============================================================================
GLOBAL_T0 = time.time()
TIME_LIMIT_SEC = int(TIME_LIMIT_H * 3600)

def run_single_experiment(model_name, seed, gpu_id):
    run_id = f"{model_name}_s{seed}"
    out_dir = f"{RESULTS_ROOT}/{run_id}"
    done_marker = f"{out_dir}/wer_output/wer_test_long_{LONG_STRATEGIES[-1]}.txt"

    if RESUME_INPUT_DIR:
        prev = sorted(glob.glob(f"{RESUME_INPUT_DIR}/**/{run_id}", recursive=True))
        if prev and not os.path.isdir(out_dir):
            shutil.copytree(prev[0], out_dir)

    if os.path.exists(done_marker):
        print(f"[GPU {gpu_id}] SKIP {run_id} (already completed).")
        return run_id, 0

    hp_path = f"{WORK}/{run_id}.yaml"
    ejust_tools.make_hparams(
        f"{REPO}/hparams/S2S/{model_name}.yaml", hp_path,
        data_folder=DATA, csv_dir=DATA, bpe_model=BPE, output_folder=out_dir,
        epochs=EPOCHS, warmup=30000, valid_search_interval=VALID_SEARCH_INTERVAL,
        batch_size=32, grad_accumulation_factor=1, num_workers=2,
        long_strategies=",".join(LONG_STRATEGIES), skip_train=False,
    )

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    cmd = [
        sys.executable, "train_egy_q1.py", hp_path,
        "--precision", "fp16", "--seed", str(seed), "--ckpt_interval_minutes", "30",
    ]
    print(f"[GPU {gpu_id}] START {run_id} -> {out_dir}")
    t0 = time.time()
    log_path = f"{WORK}/{run_id}_stdout.txt"
    with open(log_path, "w") as logf:
        proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            logf.write(line)
            if any(k in line for k in ("train_logger", "Epoch", "ERROR", "Traceback")):
                print(f"[{run_id}|GPU{gpu_id}] {line}", end="")
            if time.time() - GLOBAL_T0 > TIME_LIMIT_SEC:
                print(f"[GPU {gpu_id}] TIME LIMIT REACHED — terminating {run_id} safely (checkpoints saved).")
                proc.terminate()
                break
        proc.wait()
    print(f"[GPU {gpu_id}] DONE {run_id} | exit={proc.returncode} | {(time.time()-t0)/60:.1f} min")
    return run_id, proc.returncode

with ProcessPoolExecutor(max_workers=NUM_GPUS) as ex:
    futures = {}
    for idx, (m_name, s) in enumerate(EXPERIMENTS):
        gpu_id = idx % NUM_GPUS
        futures[ex.submit(run_single_experiment, m_name, s, gpu_id)] = (m_name, s)
    for fut in as_completed(futures):
        rid, rc = fut.result()

# ==============================================================================
# 6. OPTIONAL FAST WHISPER BASELINE (INFERENCE-ONLY ON test & test_long)
# ==============================================================================
whisper_row = None
if RUN_WHISPER_BASELINE and (time.time() - GLOBAL_T0 < TIME_LIMIT_SEC - 600):
    print(f"\n=== Running Inference-Only Baseline: {WHISPER_MODEL} ===")
    from jiwer import wer as jiwer_wer, cer as jiwer_cer
    from transformers import pipeline

    pipe = pipeline(
        "automatic-speech-recognition", model=WHISPER_MODEL,
        device=0, torch_dtype=torch.float16, chunk_length_s=30,
    )
    n_w_params = sum(p.numel() for p in pipe.model.parameters()) / 1e6
    w_stats = {"model": WHISPER_MODEL.split("/")[-1], "seed": 0, "params_m": round(n_w_params, 1)}

    for split in ["test", "test_long"]:
        df = pd.read_csv(f"{DATA}/{split}.csv", dtype=str, keep_default_na=False)
        wavs = [p.replace("$data_root", DATA) for p in df["wav"].tolist()]
        refs = df["wrd"].tolist()
        total_audio = df["duration"].astype(float).sum()
        t0 = time.time()
        outs = pipe(wavs, batch_size=16, generate_kwargs={"language": "arabic", "task": "transcribe"})
        elapsed = time.time() - t0
        hyps = [o["text"].strip() for o in outs]
        wer_key = f"{split}_wer"
        cer_key = f"{split}_cer"
        rtf_key = "rtf_test" if split == "test" else "rtf_long"
        w_stats[wer_key] = round(jiwer_wer(refs, hyps) * 100, 2)
        w_stats[cer_key] = round(jiwer_cer(refs, hyps) * 100, 2)
        w_stats[rtf_key] = round(elapsed / total_audio, 4)
        print(f"  {split:10s} | WER={w_stats[wer_key]:.2f}% | CER={w_stats[cer_key]:.2f}% | RTF={w_stats[rtf_key]:.4f}")
    whisper_row = w_stats
    del pipe; torch.cuda.empty_cache()

# ==============================================================================
# 7. Q1 STATISTICAL REPORTING (TABLES, MULTI-SEED MEAN±STD, LATEX — NO FIGURES)
# ==============================================================================
def parse_wer_file(path):
    if not os.path.exists(path):
        return None
    line = open(path, encoding="utf-8").readline()
    m = re.search(r"%WER\s+([\d\.]+)", line)
    return float(m.group(1)) if m else None

rows = [PREV_CONMAMBA_RESULT]
if whisper_row:
    rows.append(whisper_row)

long_ablation_rows = []
for m_name, seed in EXPERIMENTS:
    run_id = f"{m_name}_s{seed}"
    out_dir = f"{RESULTS_ROOT}/{run_id}"
    wdir = f"{out_dir}/wer_output"
    if not os.path.isdir(wdir):
        continue
    p_file = Path(out_dir, "params.json")
    params_m = round(json.loads(p_file.read_text())["params"] / 1e6, 1) if p_file.exists() else None
    eff_file = Path(out_dir, "efficiency.jsonl")
    rtf_map = {}
    if eff_file.exists():
        for line in eff_file.read_text().splitlines():
            rec = json.loads(line)
            if rec.get("stage") == "TEST" and rec.get("test_name"):
                rtf_map[rec["test_name"]] = rec.get("rtf")

    best_long_strat = "overlap_energy" if "overlap_energy" in LONG_STRATEGIES else LONG_STRATEGIES[-1]
    rows.append({
        "model": m_name, "seed": seed, "params_m": params_m,
        "test_wer": parse_wer_file(f"{wdir}/wer_test.txt"),
        "test_cer": parse_wer_file(f"{wdir}/cer_test.txt"),
        "test_long_wer": parse_wer_file(f"{wdir}/wer_test_long_{best_long_strat}.txt"),
        "test_long_cer": parse_wer_file(f"{wdir}/cer_test_long_{best_long_strat}.txt"),
        "test_ctc_wer": parse_wer_file(f"{wdir}/wer_ctc_test.txt"),
        "test_long_ctc_wer": parse_wer_file(f"{wdir}/wer_ctc_test_long_{best_long_strat}.txt"),
        "rtf_test": rtf_map.get("test"),
        "rtf_long": rtf_map.get(f"test_long_{best_long_strat}"),
    })
    for strat in LONG_STRATEGIES:
        long_ablation_rows.append({
            "model": m_name, "seed": seed, "strategy": strat,
            "test_long_wer": parse_wer_file(f"{wdir}/wer_test_long_{strat}.txt"),
            "test_long_cer": parse_wer_file(f"{wdir}/cer_test_long_{strat}.txt"),
            "rtf_long": rtf_map.get(f"test_long_{strat}"),
        })

df_main = pd.DataFrame(rows)
print("\n" + "=" * 90)
print("TABLE 1: PER-RUN Q1 BENCHMARK RESULTS (EJUST Egyptian Arabic ASR)")
print("=" * 90)
print(df_main.to_string(index=False))

agg_rows = []
for m, grp in df_main.groupby("model", sort=False):
    def fmt(col):
        vals = grp[col].dropna()
        if len(vals) == 0:
            return "-"
        if len(vals) == 1:
            return f"{vals.iloc[0]:.2f}"
        return f"{vals.mean():.2f} ± {vals.std(ddof=1):.2f}"
    agg_rows.append({
        "Model": m,
        "Params (M)": grp["params_m"].dropna().iloc[0] if len(grp["params_m"].dropna()) else "-",
        "Seeds": len(grp),
        "test WER (%)": fmt("test_wer"),
        "test CER (%)": fmt("test_cer"),
        "test_long WER (%)": fmt("test_long_wer"),
        "test_long CER (%)": fmt("test_long_cer"),
        "RTF (long)": f"{grp['rtf_long'].dropna().mean():.4f}" if len(grp["rtf_long"].dropna()) else "-",
    })
df_agg = pd.DataFrame(agg_rows)
print("\n" + "=" * 90)
print("TABLE 2: MULTI-SEED AGGREGATED Q1 TABLE (Mean ± Std)")
print("=" * 90)
print(df_agg.to_string(index=False))

if long_ablation_rows:
    df_long = pd.DataFrame(long_ablation_rows)
    print("\n" + "=" * 90)
    print("TABLE 3: LONG-FORM DECODING STRATEGY ABLATION (Option 2 Contribution)")
    print("=" * 90)
    print(df_long.to_string(index=False))

print("\n" + "=" * 90)
print("LATEX BOOKTABS TABLE (COPY-PASTE INTO MANUSCRIPT)")
print("=" * 90)
print(r"\begin{table}[t]")
print(r"\centering")
print(r"\caption{Egyptian Arabic ASR benchmark on EJUST (\texttt{test} and \texttt{test\_long}).}")
print(r"\begin{tabular}{lcccccc}")
print(r"\toprule")
print(r"\textbf{Model} & \textbf{Params} & \textbf{test WER} & \textbf{test CER} & \textbf{long WER} & \textbf{long CER} & \textbf{RTF} \\")
print(r"\midrule")
for _, r in df_agg.iterrows():
    model_tex = str(r["Model"]).replace("_", "\\_")
    print(f"{model_tex} & {r['Params (M)']}M & {r['test WER (%)']} & {r['test CER (%)']} & {r['test_long WER (%)']} & {r['test_long CER (%)']} & {r['RTF (long)']} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

Python 3.13.16 | PyTorch 2.11.0+cu130 | GPUs: 2x Tesla T4
Patched Mamba-ASR successfully.
Dataset root: /kaggle/input/notebooks/monaygad/egyptian-arabic-asr-masry-prepared/ejust_prepared_v1
[GPU 0] START conformer_small_s7775 -> /kaggle/working/q1_results/conformer_small_s7775[GPU 1] START conmambamamba_small_s7775 -> /kaggle/working/q1_results/conmambamamba_small_s7775

[conformer_small_s7775|GPU0] speechbrain.utils.train_logger - epoch: 1, lr: 2.18e-05, steps: 820 - train loss: 81.39 - valid loss: 68.56, valid ACC: 8.95e-02
[conmambamamba_small_s7775|GPU1] speechbrain.utils.train_logger - epoch: 1, lr: 2.18e-05, steps: 820 - train loss: 84.84 - valid loss: 68.26, valid ACC: 8.97e-02
[conformer_small_s7775|GPU0] speechbrain.utils.train_logger - epoch: 2, lr: 4.37e-05, steps: 1640 - train loss: 66.81 - valid loss: 64.00, valid ACC: 1.19e-01
[conmambamamba_small_s7775|GPU1] speechbrain.utils.train_logger - epoch: 2, lr: 4.37e-05, steps: 1640 - train loss: 65.74 - valid loss: 62.58, vali

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (

  test       | WER=71.63% | CER=33.64% | RTF=0.3120
  test_long  | WER=53.62% | CER=21.16% | RTF=0.0966

TABLE 1: PER-RUN Q1 BENCHMARK RESULTS (EJUST Egyptian Arabic ASR)
         model  seed  params_m  test_wer  test_cer  test_long_wer  test_long_cer  test_ctc_wer  test_long_ctc_wer  rtf_test  rtf_long
conmamba_small  7775      12.1     13.85      4.44          14.65           4.57         18.83              72.94       NaN       NaN
 whisper-small     0     241.7     71.63     33.64          53.62          21.16           NaN                NaN     0.312    0.0966

TABLE 2: MULTI-SEED AGGREGATED Q1 TABLE (Mean ± Std)
         Model  Params (M)  Seeds test WER (%) test CER (%) test_long WER (%) test_long CER (%) RTF (long)
conmamba_small        12.1      1        13.85         4.44             14.65              4.57          -
 whisper-small       241.7      1        71.63        33.64             53.62             21.16     0.0966

LATEX BOOKTABS TABLE (COPY-PASTE INTO MANUSCRIPT)
\